# Build GPT from Scratch

In [1]:
# import module
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1337);

## Data Preprocessing

In [2]:
# import tiny shakespare
with open("../input.txt", 'r', encoding="UTF-8") as f:
    text = f.read()

print(len(text))
print(text[:80])

1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.


In [3]:
# get all the characters in 'input.txt'
chars = sorted(set(text))
vocab_size = len(chars)

print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [4]:
# simple tokenizer:
# just simply treat a character as a token,
# and encode these character as number 0~65.

# character mapping
stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for ch, i in stoi.items()}

# tokenizer
# Using lambda is for a quick call when we want to en/de-code a sequence.
encode = lambda s: [stoi[ch] for ch in s]         # endcode : take a string, return a list of integer
decode = lambda l: ''.join([itos[i] for i in l])  # decode  : take a list of integer, return a string

In [5]:
# encode whole input text

# Use torch.long so token IDs work directly as embedding indices and cross-entropy class targets.
# Embedding api requires int32 or long/int64 dtype. Cross-entropy exactly requires long/int64.
data = torch.tensor(encode(text), dtype=torch.long)

print(data.shape, data.dtype)
print(data[:80])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8])


In [6]:
# split up data set to training and validation sets
n = int(0.9 * len(data))
tr_data = data[:n]
val_data = data[n:]

## Bigram

### input

In [7]:
# input context test
block_size = 8

x = tr_data[:block_size]
y = tr_data[1:block_size+1]

# We use different length of inputs with upper limit.
# So that whatever you input (length must under upper limit),
# the model will accept it and give you a predicition.
for i in range(block_size):
    a = (itos[j] for j in x[:i+1].tolist())
    b = itos[y[i].item()]
    print(f"when input {repr(''.join(a))}, the target will be {repr(b)}")

when input 'F', the target will be 'i'
when input 'Fi', the target will be 'r'
when input 'Fir', the target will be 's'
when input 'Firs', the target will be 't'
when input 'First', the target will be ' '
when input 'First ', the target will be 'C'
when input 'First C', the target will be 'i'
when input 'First Ci', the target will be 't'


In [11]:
# build mini-batch for batched training
batch_size = 4    # batch dimension (the number of slices in one batch, the number of parallel inputs meanwhile)
block_size = 8    # time/token dimension (the max length of a sequence of tokens)


# See the device setup cell below.
def get_batch(split):
    data = tr_data if split=='train' else val_data
    # Don't forget minus block_size on upper limit.
    ix = torch.randint(0, len(data) - block_size, (batch_size,))
    # Pytorch doesn't support by using tensor and integer to index a slice of a tensor.
    # (tensor[tensor: integer] is not suppoted.)
    # We can use comprehension to create every batch in a list,
    # then use torch.stack to stack these tensors to a big tensor.
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+1+block_size] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y


xb, yb = get_batch('train')
print(f"xb shape:{xb.shape}")
print(f"xb: {xb}")
print(f"yb shape:{yb.shape}")
print(f"yb: {yb}")
for i in range(batch_size):
    for j in range(block_size):
        a = (itos[j] for j in xb[i, :j+1].tolist())
        b = itos[yb[i, j].item()]
        print(f"when input {repr(''.join(a))}, the target will be {repr(b)}")

xb shape:torch.Size([4, 8])
xb: tensor([[57, 43, 60, 43, 52,  1, 63, 43],
        [60, 43, 42,  8,  0, 25, 63,  1],
        [56, 42,  5, 57,  1, 57, 39, 49],
        [43, 57, 58, 63,  6,  1, 58, 46]], device='cuda:0')
yb shape:torch.Size([4, 8])
yb: tensor([[43, 60, 43, 52,  1, 63, 43, 39],
        [43, 42,  8,  0, 25, 63,  1, 45],
        [42,  5, 57,  1, 57, 39, 49, 43],
        [57, 58, 63,  6,  1, 58, 46, 47]], device='cuda:0')
when input 's', the target will be 'e'
when input 'se', the target will be 'v'
when input 'sev', the target will be 'e'
when input 'seve', the target will be 'n'
when input 'seven', the target will be ' '
when input 'seven ', the target will be 'y'
when input 'seven y', the target will be 'e'
when input 'seven ye', the target will be 'a'
when input 'v', the target will be 'e'
when input 've', the target will be 'd'
when input 'ved', the target will be '.'
when input 'ved.', the target will be '\n'
when input 'ved.\n', the target will be 'M'
when input 'ved.\

### neural network

In [12]:
# create bigram model by using nn.Module
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.bi_emb = nn.Embedding(vocab_size, vocab_size)

    # __call__ will call forward here to perform the forward pass
    def forward(self, x, target=None):
        logits = self.bi_emb(x)  # shape: (B, T, C)
        
        if target is None:
            loss = None
        else:
            # cross_entropy requires shape(B, C, T) in 3 dimensions or shape(B, C) in 2 dimensions.
            # C (the channel/embedding dimension) must be the second dimension.
            # We need reshape our tensors.

            # flatten way
            # logits = logits.view(-1, logits.shape[2])
            # target = target.view(-1)
            # transpose way
            logits = logits.transpose(1, 2)
            
            loss = F.cross_entropy(logits, target)
        return logits, loss

    def generate(self, sq, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(sq)
            # Focus on the last prediction, because the predictions in context has already generated.
            # We need the last prediction to predict what is the next token should be.
            logits = logits[:, -1, :]  # return a tensor with shape(B, C)
            probs = F.softmax(logits, dim=-1) # (B, C)
            ix = torch.multinomial(probs, num_samples=1) # (B, 1)
            sq = torch.cat((sq, ix), dim=1)  # (B, T+1)
        return sq

In [13]:
# run this bigram model
m = BigramLanguageModel(vocab_size).to(device)
logits, loss = m(xb, yb)
sq = m.generate(torch.zeros((1, 1), dtype=torch.long, device=device), 15)
print(f"shape of logits: {tuple(logits.shape)}, loss: {loss.item():.4f}")
print(decode(sq[0].tolist()))

shape of logits: (4, 65, 8), loss: 4.8549

sYfmxtDkRZkNwc.


### optimizer

An optimizer updates trainable parameters using the gradients computed during backpropagation. It does not compute gradients itself.

We can easily use optimizer to update parameters.

```python
optimizer.zero_grad(set_to_none=True)  # Clear gradients from the previous step
loss.backward()                        # Compute gradients
optimizer.step()                       # Update model parameters
```

#### SGD (stochastic gradient descent) in makemore

The makemore notebooks implement vanilla gradient descent manually:

```python
for p in parameters:
    p.grad = None

loss.backward()

for p in parameters:
    p.data += -learning_rate * p.grad
```

This applies the standard update rule:

$$
\theta \leftarrow \theta - \eta \nabla_\theta L
$$

When the gradient is estimated from a randomly sampled mini-batch, this becomes mini-batch stochastic gradient descent (SGD). The neural bigram model uses full-batch gradient descent, while later makemore models use mini-batch SGD.

In [14]:
# SGD
# optimizer = torch.optim.SGD(m.parameters(), lr=lr)

#### AdamW

AdamW maintains exponential moving averages of both gradients and squared gradients, allowing it to adapt the update scale for each parameter. It also applies weight decay independently from the gradient-based update.

```python
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
)
```

Compared with the manual SGD implementation, AdamW manages its internal optimization state and parameter updates automatically. It is commonly used for training Transformer-based models.

In [15]:
# create a pytorch optimizer AdamW
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

### train

In [16]:
batch_size = 32
eval_interval = 300
max_iters = 20000

# See the estimate_loss() cell below.
for steps in range(max_iters):
    # mini-batch
    x, y = get_batch('train')

    # evaluate loss
    if steps % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {steps}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # forward pass
    logits, loss = m(x, y)
    
    # backward pass
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    
    # update
    optimizer.step()

print(loss.item())

step 0: train loss 4.6942, val loss 4.7064
step 300: train loss 4.3727, val loss 4.3752
step 600: train loss 4.0803, val loss 4.0990
step 900: train loss 3.8205, val loss 3.8367
step 1200: train loss 3.6020, val loss 3.6273
step 1500: train loss 3.4148, val loss 3.4423
step 1800: train loss 3.2577, val loss 3.2772
step 2100: train loss 3.1241, val loss 3.1347
step 2400: train loss 3.0065, val loss 3.0236
step 2700: train loss 2.9171, val loss 2.9252
step 3000: train loss 2.8292, val loss 2.8468
step 3300: train loss 2.7713, val loss 2.7739
step 3600: train loss 2.7071, val loss 2.7292
step 3900: train loss 2.6737, val loss 2.6825
step 4200: train loss 2.6364, val loss 2.6489
step 4500: train loss 2.6010, val loss 2.6098
step 4800: train loss 2.5780, val loss 2.5906
step 5100: train loss 2.5645, val loss 2.5796
step 5400: train loss 2.5485, val loss 2.5662
step 5700: train loss 2.5417, val loss 2.5395
step 6000: train loss 2.5288, val loss 2.5220
step 6300: train loss 2.5161, val loss 2

In [17]:
# generate something after training
sq = m.generate(torch.zeros((1, 1), dtype=torch.long, device=device), 300)
print(decode(sq[0].tolist()))


windakis by bth

Hiset bube d e.
S:
O:
IS:
Falatanss:
Wanthar u qur, vet?
F dilasoate awice my.

Hastarom oroup
Yowhthetof isth ble mil ndill, ath iree sengmin lat Heriliovets, and Win nghirileranousel lind me l.
HAshe ce hiry:
Supr aisspllw y.
Herindu n Boopetelaves
MP:

Pl, d mothakleo Windo whth 


### some optimizations

In [18]:
# Set device to use GPU if have GPU.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# Send training data and model to device (GPU), then we can use gpu to accelerate the compute.
# Only move the model and tensors involved in GPU computation,
# such as training batches, to the GPU; preprocessing and other work can remain on the CPU.
# We just need move mini-batch in training to device, here is just a demostrasion.
# m = m.to(device)
# xb = xb.to(device)
# yb = yb.to(device)

In [19]:
# estimate average loss
# a hyperparameter for controlling the round of evaluatation
eval_iters = 200

@torch.no_grad()
def estimate_loss():
    # use a dict to store every loss of each date set
    out = {}
    # switch to evaluation mode
    m.eval()
    # evaluate train set and val set
    for split in ['train', 'val']:
        # losses storage 200 losses for caculating average loss
        losses = torch.zeros(eval_iters)
        # caculate loss and store
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = m(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    # switch to training mode
    m.train()
    return out

## Transformer

### The mathematical trick in self-attention

In [21]:
# For each token, its new representation is computed by averaging its own features with all preceding tokens.
# This (aggregating historical context) serves as a vital stepping stone to understanding self-attention.

# toy example
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [22]:
# version 1
# We want x[b, t] = mean_{ i<=t | x[b, i] }
xbow0 = torch.zeros(B, T, C)  # bow means bag of words
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow0[b, t] = torch.mean(xprev, dim=0)

In [23]:
# version 2
# using matrix multiplication to replace nested loops

# torch.tril return lower triangular part of tensor, and the other part is zero
wei = torch.tril(torch.ones(T, T))  # wei is short for weights
wei = wei / wei.sum(dim=1, keepdim=True)  # similar to normalization, get weight on every position
xbow1 = wei @ x  # (T, T) @ (B, T, C) ---> (B, T, C)
# Multi-dimensional matmul semantics:
# <= 2D tensors (excluding scalars): Standard matrix multiplication.
# > 2D tensors: Treated as a batch of matrices. Here, it effectively performs B
#   independent matrix multiplications (Batch MatMul) and stacks the results.

# This way equivalent to the naive nested loops (computing the weighted sums directly), 
# but significantly faster. This vectorization leverages PyTorch's broadcasting 
# semantics and highly optimized matmul kernels under the hood.
torch.allclose(xbow0, xbow1)

True

In [24]:
# version 3: use softmax
tril = torch.tril(torch.ones(T, T))

# The following operations are mathematically equivalent to version 2.
# But it aligns better with the exponential function used during training or inference.
# Feel free to step through them line-by-line to inspect the intermediate tensors.
# Hint: e^0 == 1; e^(-inf) == 0
wei = torch.zeros(T, T)
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=1)

xbow2 = wei @ x

torch.allclose(xbow0, xbow2)

True